# 🚀 TOEIC Reading Passages High-Speed Translator (Qwen2.5-7B on Google Colab GPU)
Notebook này dùng để dịch song ngữ chuẩn theo từng câu (**Pre-aligned Sentence Pairs**) cho **Reading Passages (Part 6 & 7)** trên GPU **Tesla T4 (16GB VRAM)** miễn phí của Google Colab.

### ⚡ Tại sao chạy trên Colab T4 nhanh hơn gấp 5 - 10 lần?
- **16GB VRAM**: Nạp **100% layer (29/29 layer)** của model `qwen2.5:7b-instruct-q4_K_M` lên VRAM.
- Không bị nghẽn RAM bus giữa CPU và GPU.
- Tốc độ sinh: **~50 - 70 tokens/giây** (thời gian dịch mỗi passage chỉ còn **~3 - 6 giây**, toàn bộ 178 passage chỉ mất khoảng **10 - 15 phút** thay vì 1.5 - 2.5 giờ trên laptop).

---

### 📋 Quy trình 5 bước đơn giản:
1. **Bật GPU T4**: Chọn menu **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** -> Nhấn **Save**.
2. **Chạy Cell 1 & 2**: Cài đặt Ollama và tải model `qwen2.5:7b-instruct-q4_K_M` (khoảng 1 - 2 phút).
3. **Chạy Cell 3**: Upload file `untranslated_passages.json` (được xuất từ laptop bằng lệnh: `python scripts/export_untranslated_passages.py`).
4. **Chạy Cell 4**: Tiến hành dịch tự động toàn bộ passage với tiến trình `tqdm` và cơ chế auto-save checkpoint liên tục.
5. **Chạy Cell 5**: Tải file `translated_passages.json` về máy tính và import vào MySQL bằng lệnh:
   ```bash
   python scripts/import_translated_passages.py
   ```


In [ ]:
# BƯỚC 1: Kiểm tra phần cứng GPU T4
!nvidia-smi

In [ ]:
# BƯỚC 2: Cài đặt Ollama và các thư viện cần thiết
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama pydantic tqdm requests -q

In [ ]:
# BƯỚC 3: Khởi động Ollama daemon trong background & tải model Qwen2.5 7B
import subprocess
import time
import requests

print("[*] Đang khởi động Ollama daemon trong background...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

try:
    resp = requests.get("http://127.0.0.1:11434/api/tags")
    print(f"[+] Ollama daemon đã sẵn sàng (HTTP {resp.status_code})")
except Exception as e:
    print(f"[!] Cảnh báo kết nối Ollama: {e}")

print("\n[*] Đang tải model qwen2.5:7b-instruct-q4_K_M (khoảng 4.7GB, mạng Colab tải chỉ mất ~1-2 phút)...")
!ollama pull qwen2.5:7b-instruct-q4_K_M
print("\n[+] Đã tải xong model và sẵn sàng chạy GPU!")

In [ ]:
# BƯỚC 4: Upload file untranslated_passages.json từ máy tính của bạn
import os
import json
from google.colab import files

print("Nhấn nút 'Choose Files' bên dưới để tải lên file untranslated_passages.json:")
uploaded = files.upload()

input_filename = None
for fname in uploaded.keys():
    if fname.endswith(".json"):
        input_filename = fname
        break

if not input_filename:
    raise FileNotFoundError("Chưa có file JSON nào được tải lên!")

with open(input_filename, "r", encoding="utf-8") as f:
    passages_data = json.load(f)

total_items = sum(len(p.get("items", [])) for p in passages_data)
print(f"\n[+] Đã nạp thành công {len(passages_data)} passages ({total_items:,} câu) từ file '{input_filename}'.")

In [ ]:
# BƯỚC 5: Pipeline dịch tự động song ngữ tốc độ cao (100% GPU T4)
import json
import re
import time
import os
from typing import List, Dict, Any, Optional
import requests
from pydantic import BaseModel
from tqdm.notebook import tqdm

MODEL_NAME = "qwen2.5:7b-instruct-q4_K_M"
OLLAMA_URL = "http://127.0.0.1:11434"
CHUNK_SIZE = 10
CHECKPOINT_INTERVAL = 10
OUTPUT_FILE = "translated_passages.json"
CHECKPOINT_FILE = "translated_passages_checkpoint.json"

class TranslationPair(BaseModel):
    sentence_id: int
    vi: str

class BatchTranslationResponse(BaseModel):
    pairs: List[TranslationPair]

def chunk_items(items: List[Dict[str, Any]], chunk_size: int = 10) -> List[List[Dict[str, Any]]]:
    return [items[i:i + chunk_size] for i in range(0, len(items), chunk_size)]

def translate_single_chunk(
    chunk: List[Dict[str, Any]],
    passage_title: str,
    max_retries: int = 3,
) -> Optional[Dict[int, str]]:
    translation_payload = [
        {"sentence_id": it["sentence_id"], "en": it["en"]}
        for it in chunk
    ]
    prompt = f"""You are an expert English-to-Vietnamese translator specializing in TOEIC test reading passages.
Context/Title: "{passage_title}"
Translate each numbered English item below into accurate, natural, professional Vietnamese suitable for Vietnamese learners preparing for the TOEIC exam.

CRITICAL RULES:
1. Return EXACTLY 1 Vietnamese translation for each item, with the EXACT same sentence_id.
2. Keep the exact count of items ({len(chunk)} items in, {len(chunk)} items out). Do NOT merge sentences, split sentences, or skip any items.
3. Preserve all placeholder tokens like '______[139]', '------- (131)', '[1]', '— [133] —', URLs, phone numbers, and email addresses in the translation at the grammatically natural position.
4. For standalone placeholders (e.g. '______[141].' or '— [133] —.'), output the placeholder as is.
5. Translate metadata labels appropriately (e.g. 'To:' -> 'Đến:', 'From:' -> 'Từ:', 'Subject:' -> 'Chủ đề:', 'Date:' -> 'Ngày:', 'Re:' -> 'V/v:').

Output format MUST be valid JSON with this exact structure:
{{"pairs": [{{"sentence_id": 1, "vi": "Bản dịch câu 1"}}, ...]}}

Items to translate:
{json.dumps(translation_payload, ensure_ascii=False, indent=2)}
"""
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "format": "json",
        "stream": False,
        "options": {
            "temperature": 0.1,
            "num_gpu": 99,       # Offload toàn bộ 29 layers lên GPU T4 (16GB VRAM)
            "num_ctx": 1024,
            "num_batch": 512,
        },
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=60)
            if resp.status_code != 200:
                time.sleep(1)
                continue
            data = resp.json()
            raw_content = data.get("message", {}).get("content", "").strip()
            clean_content = raw_content
            if clean_content.startswith("```"):
                clean_content = re.sub(r"^```(?:json)?\s*", "", clean_content)
                clean_content = re.sub(r"\s*```$", "", clean_content)

            res = BatchTranslationResponse.model_validate_json(clean_content)
            vi_map = {p.sentence_id: p.vi.strip() for p in res.pairs if p.vi.strip()}
            if all(it["sentence_id"] in vi_map for it in chunk):
                return vi_map
        except Exception:
            time.sleep(1)
            continue
    return None

def process_passage(passage: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    pid = passage["id"]
    title = passage.get("title", "")
    items = passage.get("items", [])
    if not items:
        return None

    overall_vi_map = {}
    chunks = chunk_items(items, chunk_size=CHUNK_SIZE)
    for chunk in chunks:
        vi_map = translate_single_chunk(chunk, title)
        if vi_map:
            overall_vi_map.update(vi_map)
        else:
            # Subdivide retry nếu chunk 10 câu gặp lỗi
            subchunks = chunk_items(chunk, max(1, len(chunk) // 2))
            for sub in subchunks:
                sub_map = translate_single_chunk(sub, title)
                if sub_map:
                    overall_vi_map.update(sub_map)
                else:
                    for single in sub:
                        s_map = translate_single_chunk([single], title)
                        if s_map:
                            overall_vi_map.update(s_map)

    if not all(it["sentence_id"] in overall_vi_map for it in items):
        return None

    sentence_pairs = []
    for it in items:
        sid = it["sentence_id"]
        sentence_pairs.append({
            "sentence_id": sid,
            "en": it["en"],
            "vi": overall_vi_map[sid],
            "question_numbers": it.get("question_numbers", []),
            "type": it.get("type", "text"),
            "doc_index": it.get("doc_index", 0),
            "translated_by": "ollama-qwen2.5-colab",
        })

    return {
        "id": pid,
        "test_id": passage.get("test_id"),
        "part": passage.get("part"),
        "title": title,
        "passage_sentence_pairs": sentence_pairs,
    }

# Khôi phục tiến trình từ checkpoint nếu từng bị ngắt quãng
translated_results = []
processed_ids = set()
if os.path.exists(CHECKPOINT_FILE):
    try:
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            translated_results = json.load(f)
            processed_ids = {r["id"] for r in translated_results}
            print(f"[*] Đã khôi phục checkpoint: {len(translated_results)} passages đã hoàn tất trước đó.")
    except Exception:
        translated_results = []
        processed_ids = set()

to_process = [p for p in passages_data if p["id"] not in processed_ids]
print(f"[*] Số passages cần dịch tiếp: {len(to_process)} | Đã hoàn tất: {len(processed_ids)}")

start_time = time.time()
success_count = 0
fail_count = 0

with tqdm(total=len(to_process), desc="Tiến trình dịch GPU", unit="passage") as pbar:
    for idx, passage in enumerate(to_process):
        res = process_passage(passage)
        if res:
            translated_results.append(res)
            success_count += 1
        else:
            fail_count += 1
            print(f" [!] Lỗi dịch passage: {passage['id']}")

        pbar.update(1)

        # Tự động lưu checkpoint định kỳ 10 passage / lần
        if (idx + 1) % CHECKPOINT_INTERVAL == 0 or (idx + 1) == len(to_process):
            with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
                json.dump(translated_results, f, ensure_ascii=False, indent=2)

# Lưu file kết quả hoàn chỉnh
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(translated_results, f, ensure_ascii=False, indent=2)

elapsed = time.time() - start_time
print(f"\n==================================================")
print(f"Hoàn tất toàn bộ trong: {elapsed / 60:.1f} phút ({elapsed:.1f} giây)!")
print(f"Thành công: {success_count} | Thất bại: {fail_count} | Tổng kết quả: {len(translated_results)}")
print(f"Tốc độ trung bình: {elapsed / max(1, success_count):.2f}s / passage")
print(f"==================================================")

In [ ]:
# BƯỚC 6: Kiểm tra mẫu kết quả & Tự động tải file translated_passages.json về máy tính
import json
from google.colab import files

with open("translated_passages.json", "r", encoding="utf-8") as f:
    results = json.load(f)

print(f"[+] Tổng số passage đã dịch: {len(results)}")
if results:
    sample = results[0]
    print(f"\n--- Xem trước passage mẫu: {sample.get('title')} ---")
    for pair in sample.get("passage_sentence_pairs", [])[:3]:
        print(f"[#{pair['sentence_id']}] EN: {pair['en']}")
        print(f"       VI: {pair['vi']}\n")

print("[*] Đang kích hoạt tải file translated_passages.json về máy tính của bạn...")
files.download("translated_passages.json")
print("[+] Sau khi tải file về, hãy đặt vào thư mục 'server' và chạy: python scripts/import_translated_passages.py")